In [1]:
import time
import os
import logging
import argparse
import random
import json
import torch
import numpy as np
import pandas as pd

from transformers import GenerationConfig
from torch.utils.data import DataLoader
from pynvml import *
import psutil
from scripts.llama_child import Llama
from scripts.mistral_child import Mistral
from utils.misc_utils import *
from utils.calculate_eval_metrics import calculate_bleu, calculate_rouge, calculate_exact_string_match, calculate_sequence_accuracy


/mfs1/u/jayje/condaenvs/test_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-04-14 11:46:11,565] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/mfs1/u/jayje/condaenvs/test_env/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/mfs1/u/jayje/condaenvs/test_env/compiler_compat/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


In [19]:
device = 'cuda'

In [2]:

cls_params =  {
                        "model_name": "Kawon/llama3.1-food-finetune_v14_r8",
                        "dataset_name": "super_glue",
                        "data_size":None,
                        "task":"copa",
                        "use_quantized":False,
                        "use_peft":False,
                        "peft_method":None,
                        "r":None,
                        "density":None,
                        "seed":123
                    }

In [3]:
model_cls = Llama(**cls_params)

In [4]:
prompt = model_cls.load_task_prompt()
target_key = prompt['assistant_output']
answer_len = prompt['answer_len']
assistant_start_token=model_cls.assistant_start_token
eos_token=model_cls.eos_token

In [5]:
model, tokenizer = model_cls.get_model_and_tokenizer(
    tokenizer_path="meta-llama/Llama-3.1-8B-Instruct",
    use_flash_attention=True, use_safetensors=True, on_vector=False)

tokenizer.padding_side = 'left' # important for batch generation

using args:  {'pretrained_model_name_or_path': 'Kawon/llama3.1-food-finetune_v14_r8', 'attn_implementation': 'flash_attention_2', 'use_safetensors': True, 'torch_dtype': torch.bfloat16}


You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 50.15it/s]


Padding tokenizer pad token to 128009


In [7]:
model.eval()
model.to("cuda")

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.1, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear(
            (base_layer): Linear(in_features=4096, out_features=1024, bias=False)
            (lora_dropout): ModuleDict(
              (default): 

In [11]:
!nvidia-smi

Mon Apr 14 11:49:18 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.78                 Driver Version: 550.78         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:B1:00.0 Off |                  Off |
| 30%   35C    P8             21W /  300W |   15720MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
data = model_cls.get_data(tokenizer=tokenizer, max_seq_len=2048, filter_long_seq=False)

Map: 100%|██████████| 40/40 [00:00<00:00, 1623.47 examples/s]

Data train size : 360 Validation: 72 Test: 40


In [13]:
mycollator=model_cls.get_datacollator(tokenizer=tokenizer, completion_only=True)

In [14]:
def get_dataloader(data, split, tokenizer, batch_size, mycollator, max_seq_len, assistant_start_token):
    """Get dataloader for batched evaluation
    """

    model_inputs = []
    # convert 'text' to tokens
    for i in range(len(data[split])):
        prompt = data[split][i]['text'].split(assistant_start_token)[0] + f"\n {assistant_start_token}"
        tok_dict = tokenizer(prompt, add_special_tokens=False)
        if len(tok_dict['input_ids']) < max_seq_len:
            # to retrieve grountruth label in eval_on_heldout
            tok_dict['data_idx'] = i 
            model_inputs.append(tok_dict)

    loader = DataLoader(
            model_inputs,
            collate_fn=mycollator,
            batch_size=batch_size,
            shuffle=False
        )
    return loader

In [15]:
myloader = get_dataloader(data=data, split='test', tokenizer=tokenizer, batch_size=8,
                              mycollator=mycollator, max_seq_len=2048,
                              assistant_start_token=model_cls.assistant_start_token
                              )

In [24]:
def eval_on_heldout(model, data, split, tokenizer, dataloader, answer_len, target_key,
                    assistant_start_token, eos_token):
    """Evaluate the loaded model on the task dataset.
    """

    d_res={'target': [], 'pred_clean': []}
    result=None

    if answer_len == "short":
        max_new_toks = 30
    elif answer_len == "long":
        max_new_toks = 500    
    
    # TODO: this generation config may have to change for non-clsasification task
    generation_config = GenerationConfig(top_k=2,
                                                temperature=0.1,
                                                max_new_tokens=max_new_toks,
                                                pad_token_id=tokenizer.pad_token_id,
                                                num_beam=4,
                                                do_sample=True
        )

    with torch.no_grad():
        for batch in dataloader:
            batch_in = {k: v.to(device) for k, v in batch.items() if k in ['input_ids','attention_mask']}
            outputs = model.generate(**batch_in, generation_config=generation_config)
            response = tokenizer.batch_decode(outputs)
            response_clean = [clean_model_output(out=r, assistant_start_token=assistant_start_token, eos_token=eos_token) for r in response]
            target_clean = [clean_target_output(data[split][int(idx)][target_key]) for idx in batch['data_idx']]
            d_res['target'] += target_clean
            d_res['pred_clean'] += response_clean
    
    result = pd.DataFrame(d_res)
    result['target'] = result['target'].astype(str)
    result['pred_clean'] = result['pred_clean'].astype(str)

    return result


def clean_model_output(out, assistant_start_token, eos_token):
    """Extract final model answer from model generation
    """
    
    out = out.split(assistant_start_token)[-1]
    out = out.split('####')[-1] # a few datasets (gsm8k) with #### as the final answer designator; TODO: shouldn't affect other tasks; remove?
    out = out.strip('\n').split('\n')[0] # take the very first answer before the line change
    out = out.replace(eos_token,'').lower() # misc cleaning: remove end of sequence tok, remove space, lower case
    out = out.strip('.').strip() # remove '.' in case of short answers
    
    return out

def clean_target_output(tar):
    """Extract final target answer from task dataset
    """
    
    tar = str(tar)
    tar = tar.split('####')[-1]
    tar = tar.strip('.').strip().lower()

    return tar


In [25]:
%%time
eval_result = eval_on_heldout(model=model, data=data, split='test', tokenizer=tokenizer,answer_len=answer_len,dataloader=myloader,
                                 target_key=target_key, assistant_start_token=model_cls.assistant_start_token, eos_token=model_cls.eos_token
                                 )

CPU times: user 2.1 s, sys: 9.95 ms, total: 2.11 s
Wall time: 2.11 s


In [28]:
sum(eval_result['target']==eval_result['pred_clean'])/eval_result.shape[0]

0.675

/mfs1/u/jayje/condaenvs/multi_env/bin/nvcc


In [ ]:

model.eval()
model.to(device)
# logger.info("device for evaluation: ", set([p.device for p in model.parameters()]))